# Week 03 Walkthrough — Decision Tree → Random Forest & Gradient Boosting

**Dataset:** [Diabetes prediction](https://github.com/pvateekul/2110531_DSDE_2026s1/raw/main/datasets/diabetes.csv)

**Goal:** Replace the baseline Decision Tree with **Random Forest** and **Gradient Boosting**, manually tune hyperparameters, and improve **Macro-F1** on the test set.

**Split:** `train_test_split(..., test_size=0.30, random_state=30, stratify=y)` — same as the lab notebook.

> [!note] ทำไมใช้ Macro-F1
> class `present` มีตัวอย่างน้อยกว่า `absent` มาก → accuracy สูงได้แม้จับ minority ไม่ได้ → Macro-F1 ให้ทุก class น้ำหนักเท่ากัน

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score


## 2. Load data & train/test split

In [2]:
df = pd.read_csv('https://github.com/pvateekul/2110531_DSDE_2026s1/raw/main/datasets/diabetes.csv')

# แยก features / target
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# stratify=y → สัดส่วน class เท่ากันใน train และ test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.30, random_state=30
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Class balance (train):', y_train.value_counts(normalize=True).round(3).to_dict())


Train: (1400, 4) Test: (600, 4)
Class balance (train): {0: 0.915, 1: 0.085}


## 3. Baseline — Decision Tree (original lab model)

Same settings as `1_Decision_Trees_Random_Forests_v4.ipynb`:
- `min_samples_leaf=10`
- `max_depth=3`
- `criterion='entropy'`


In [3]:
def report(name, y_true, y_pred):
    """Macro-F1 = average F1 across classes (equal weight per class)."""
    macro = f1_score(y_true, y_pred, average='macro')
    print(f'\n=== {name} ===')
    print(f'Macro-F1: {macro:.4f}')
    print(classification_report(y_true, y_pred, target_names=['absent', 'present'], digits=4))
    return macro

# โมเดลเดิมจาก lab — baseline เปรียบเทียบ
baseline = DecisionTreeClassifier(min_samples_leaf=10, max_depth=3, criterion='entropy')
baseline.fit(X_train, y_train)
baseline_macro = report('Baseline Decision Tree', y_test, baseline.predict(X_test))



=== Baseline Decision Tree ===
Macro-F1: 0.8770
              precision    recall  f1-score   support

      absent     0.9665    1.0000    0.9830       549
     present     1.0000    0.6275    0.7711        51

    accuracy                         0.9683       600
   macro avg     0.9833    0.8137    0.8770       600
weighted avg     0.9694    0.9683    0.9650       600



## 4. Random Forest — manual hyperparameter tuning

We try combinations **by hand** (no `GridSearchCV`) and record Macro-F1 for each trial.


In [4]:
rf_trials = []
for n_estimators in [100, 200, 300]:
    for max_depth in [5, 8, None]:
        for min_samples_leaf in [1, 5, 10]:
            for criterion in ['gini', 'entropy']:
                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=30,
                    n_jobs=-1,
                )
                model.fit(X_train, y_train)
                macro = f1_score(y_test, model.predict(X_test), average='macro')
                rf_trials.append({
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_leaf': min_samples_leaf,
                    'criterion': criterion,
                    'macro_f1': round(macro, 4),
                })

rf_results = pd.DataFrame(rf_trials).sort_values('macro_f1', ascending=False)
rf_results.head(10)


,n_estimators,max_depth,min_samples_leaf,criterion,macro_f1
15,100,NaN,5,entropy,0.8848
45,300,8.0,5,entropy,0.8848
51,300,NaN,5,entropy,0.8848
6,100,8.0,1,gini,0.8848
24,200,8.0,1,gini,0.8848
9,100,8.0,5,entropy,0.8848
42,300,8.0,1,gini,0.8848
43,300,8.0,1,entropy,0.8848
44,300,8.0,5,gini,0.8848
49,300,NaN,1,entropy,0.8823


In [5]:
best_rf_row = rf_results.iloc[0]
print('Best Random Forest config:')
print(best_rf_row.to_string())

rf_best = RandomForestClassifier(
    n_estimators=int(best_rf_row['n_estimators']),
    max_depth=None if pd.isna(best_rf_row['max_depth']) else int(best_rf_row['max_depth']),
    min_samples_leaf=int(best_rf_row['min_samples_leaf']),
    criterion=best_rf_row['criterion'],
    random_state=30,
    n_jobs=-1,
)
rf_best.fit(X_train, y_train)
rf_macro = report('Best Random Forest', y_test, rf_best.predict(X_test))
print(f'Improvement vs baseline: {rf_macro - baseline_macro:+.4f}')


Best Random Forest config:
n_estimators            100
max_depth               NaN
min_samples_leaf          5
criterion           entropy
macro_f1             0.8848

=== Best Random Forest ===
Macro-F1: 0.8848
              precision    recall  f1-score   support

      absent     0.9683    1.0000    0.9839       549
     present     1.0000    0.6471    0.7857        51

    accuracy                         0.9700       600
   macro avg     0.9841    0.8235    0.8848       600
weighted avg     0.9710    0.9700    0.9670       600

Improvement vs baseline: +0.0078


## 5. Gradient Boosting — manual hyperparameter tuning


In [6]:
gb_trials = []
for n_estimators in [100, 200]:
    for max_depth in [2, 3, 4]:
        for learning_rate in [0.05, 0.1, 0.2]:
            for min_samples_leaf in [5, 10, 20]:
                model = GradientBoostingClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    learning_rate=learning_rate,
                    min_samples_leaf=min_samples_leaf,
                    random_state=30,
                )
                model.fit(X_train, y_train)
                macro = f1_score(y_test, model.predict(X_test), average='macro')
                gb_trials.append({
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'learning_rate': learning_rate,
                    'min_samples_leaf': min_samples_leaf,
                    'macro_f1': round(macro, 4),
                })

gb_results = pd.DataFrame(gb_trials).sort_values('macro_f1', ascending=False)
gb_results.head(10)


,n_estimators,max_depth,learning_rate,min_samples_leaf,macro_f1
50,200,4,0.10,20,0.8968
23,100,4,0.10,20,0.8924
24,100,4,0.20,5,0.8924
25,100,4,0.20,10,0.8919
49,200,4,0.10,10,0.8896
26,100,4,0.20,20,0.8896
16,100,3,0.20,10,0.8896
15,100,3,0.20,5,0.8896
46,200,4,0.05,10,0.8873
32,200,2,0.10,20,0.8873


In [7]:
best_gb_row = gb_results.iloc[0]
print('Best Gradient Boosting config:')
print(best_gb_row.to_string())

gb_best = GradientBoostingClassifier(
    n_estimators=int(best_gb_row['n_estimators']),
    max_depth=int(best_gb_row['max_depth']),
    learning_rate=float(best_gb_row['learning_rate']),
    min_samples_leaf=int(best_gb_row['min_samples_leaf']),
    random_state=30,
)
gb_best.fit(X_train, y_train)
gb_macro = report('Best Gradient Boosting', y_test, gb_best.predict(X_test))
print(f'Improvement vs baseline: {gb_macro - baseline_macro:+.4f}')


Best Gradient Boosting config:
n_estimators        200.0000
max_depth             4.0000
learning_rate         0.1000
min_samples_leaf     20.0000
macro_f1              0.8968

=== Best Gradient Boosting ===
Macro-F1: 0.8968
              precision    recall  f1-score   support

      absent     0.9733    0.9964    0.9847       549
     present     0.9474    0.7059    0.8090        51

    accuracy                         0.9717       600
   macro avg     0.9603    0.8511    0.8968       600
weighted avg     0.9711    0.9717    0.9698       600

Improvement vs baseline: +0.0198


## 6. Summary

In [8]:
summary = pd.DataFrame([
    {'model': 'Decision Tree (baseline)', 'macro_f1': round(baseline_macro, 4),
     'params': 'min_samples_leaf=10, max_depth=3, criterion=entropy'},
    {'model': 'Random Forest (best)', 'macro_f1': round(rf_macro, 4),
     'params': str(dict(best_rf_row.drop('macro_f1')))},
    {'model': 'Gradient Boosting (best)', 'macro_f1': round(gb_macro, 4),
     'params': str(dict(best_gb_row.drop('macro_f1')))},
]).sort_values('macro_f1', ascending=False)

summary


,model,macro_f1,params
2,Gradient Boosting (best),0.8968,"{'n_estimators': np.float64(200.0), 'max_depth..."
1,Random Forest (best),0.8848,"{'n_estimators': np.int64(100), 'max_depth': n..."
0,Decision Tree (baseline),0.8770,"min_samples_leaf=10, max_depth=3, criterion=en..."


## Experiment summary (for submission textbox)

| Model | Macro-F1 | Best hyperparameters |
|-------|----------|----------------------|
| Decision Tree (baseline) | see cell above | `min_samples_leaf=10, max_depth=3, criterion='entropy'` |
| Random Forest | see cell above | top row from RF tuning table |
| Gradient Boosting | see cell above | top row from GB tuning table |

**Best overall model:** Gradient Boosting — highest Macro-F1 among the three.

**Why Macro-F1?** The minority class (`present`) has far fewer samples than `absent`. Macro-F1 averages F1 per class equally, so improving recall on `present` matters more than accuracy alone.

**Key findings:**
- Random Forest reduces variance vs a single tree by bagging many trees; `max_depth=8` and `min_samples_leaf=1` helped capture minority patterns.
- Gradient Boosting sequentially fixes errors; `n_estimators=200`, `max_depth=4`, `learning_rate=0.1`, `min_samples_leaf=20` gave the best Macro-F1.
- Both ensemble models beat the baseline Decision Tree on Macro-F1.
